# Chapter 16 — Moderate Problems

Twenty-six mixed problems. Each has a paraphrased prompt, a solution with complexity notes, and tests. Where it helps, a brute-force reference checks the optimized version.

In [1]:
from __future__ import annotations

import math
import random
import re
from collections import Counter, OrderedDict, defaultdict, deque
from dataclasses import dataclass
from fractions import Fraction
from functools import cache

## 16.1 Number Swapper
Swap two numbers in place, with no temporary variable. The arithmetic trick (`a = a - b`, ...) and the XOR trick both work. Python's tuple swap is the idiomatic way.

In [2]:
def swap_arith(a: int, b: int) -> tuple[int, int]:
    a = a - b
    b = a + b   # b = original a
    a = b - a   # a = original b
    return a, b


def swap_xor(a: int, b: int) -> tuple[int, int]:
    a ^= b
    b ^= a
    a ^= b
    return a, b


assert swap_arith(3, -9) == swap_xor(3, -9) == (-9, 3)
a, b = 1, 2; a, b = b, a; assert (a, b) == (2, 1)
print("16.1 ok")

16.1 ok


## 16.2 Word Frequencies
Find how often a word appears in a book. For repeated queries, preprocess once into a hash table (`Counter`), so each lookup is `O(1)`.

In [3]:
class WordFrequencies:
    def __init__(self, text: str) -> None:
        self.counts = Counter(re.findall(r"[a-z']+", text.lower()))

    def frequency(self, word: str) -> int:
        return self.counts[word.lower()]


wf = WordFrequencies("The cat and the hat. THE end.")
assert wf.frequency("the") == 3 and wf.frequency("dog") == 0
print("16.2 ok")

16.2 ok


## 16.3 Intersection
Given two line segments by their endpoints, return their intersection point, if any. Use exact `Fraction` arithmetic. Solve the parametric equations. Handle parallel and collinear-overlap cases (for an overlap we return one shared point).

In [4]:
Point = tuple[Fraction, Fraction]


def intersection(p1, p2, p3, p4) -> Point | None:
    (x1, y1), (x2, y2), (x3, y3), (x4, y4) = [(Fraction(x), Fraction(y)) for x, y in (p1, p2, p3, p4)]
    d = (x2 - x1) * (y4 - y3) - (y2 - y1) * (x4 - x3)
    if d == 0:  # parallel
        if (x3 - x1) * (y2 - y1) != (y3 - y1) * (x2 - x1):
            return None  # parallel, not collinear
        # collinear: project onto the dominant axis, check overlap
        key = (lambda p: p[0]) if x1 != x2 or x3 != x4 else (lambda p: p[1])
        s1 = sorted([(x1, y1), (x2, y2)], key=key)
        s2 = sorted([(x3, y3), (x4, y4)], key=key)
        start = max(s1[0], s2[0], key=key)
        end = min(s1[1], s2[1], key=key)
        return start if key(start) <= key(end) else None
    t = ((x3 - x1) * (y4 - y3) - (y3 - y1) * (x4 - x3)) / d
    u = ((x3 - x1) * (y2 - y1) - (y3 - y1) * (x2 - x1)) / d
    if 0 <= t <= 1 and 0 <= u <= 1:
        return (x1 + t * (x2 - x1), y1 + t * (y2 - y1))
    return None


assert intersection((0, 0), (4, 4), (0, 4), (4, 0)) == (2, 2)
assert intersection((0, 0), (1, 1), (2, 2), (3, 5)) is None
assert intersection((0, 0), (2, 2), (1, 1), (3, 3)) == (1, 1)     # collinear overlap
assert intersection((0, 0), (1, 1), (2, 2), (3, 3)) is None        # collinear, disjoint
assert intersection((0, 0), (0, 2), (1, 0), (1, 2)) is None        # parallel vertical
print("16.3 ok")

16.3 ok


## 16.4 Tic Tac Win
Decide whether someone has won a game of tic-tac-toe on an N×N board. Check the rows, columns, and both diagonals: `O(N²)`. If the check runs after every move, only the last move's row, column, and diagonals need checking: `O(N)`.

In [5]:
def tic_tac_winner(board: list[str]) -> str | None:
    n = len(board)
    lines = [board[r] for r in range(n)]
    lines += ["".join(board[r][c] for r in range(n)) for c in range(n)]
    lines.append("".join(board[i][i] for i in range(n)))
    lines.append("".join(board[i][n - 1 - i] for i in range(n)))
    for line in lines:
        if line[0] != " " and line == line[0] * n:
            return line[0]
    return None


def won_by_last_move(board: list[str], r: int, c: int) -> bool:
    n, p = len(board), board[r][c]
    return (all(board[r][i] == p for i in range(n)) or all(board[i][c] == p for i in range(n))
            or (r == c and all(board[i][i] == p for i in range(n)))
            or (r + c == n - 1 and all(board[i][n - 1 - i] == p for i in range(n))))


assert tic_tac_winner(["XOX", "OXO", "OXX"]) == "X"
assert tic_tac_winner(["XOX", "XOO", "OXX"]) is None
assert tic_tac_winner(["OOOO", "XX X", "X  X", "   X"]) == "O"
assert won_by_last_move(["XOX", "OXO", "OXX"], 2, 2)
print("16.4 ok")

16.4 ok


## 16.5 Factorial Zeros
Count the trailing zeros of n!. Each zero comes from a factor of 10 = 2·5, and 5s are the scarcer factor. The count is `n//5 + n//25 + n//125 + …`.

In [6]:
def factorial_zeros(n: int) -> int:
    count, p = 0, 5
    while p <= n:
        count += n // p
        p *= 5
    return count


for n in range(200):
    s = str(math.factorial(n))
    assert factorial_zeros(n) == len(s) - len(s.rstrip("0"))
print("16.5 ok, 1000! has", factorial_zeros(1000), "trailing zeros")

16.5 ok, 1000! has 249 trailing zeros


## 16.6 Smallest Difference
Given two arrays, find the pair (one value from each) with the smallest non-negative difference. Sort both, then walk them with two pointers, always advancing the smaller value: `O(A log A + B log B)`.

In [7]:
def smallest_difference(a: list[int], b: list[int]) -> int:
    a, b = sorted(a), sorted(b)
    i = j = 0
    best = math.inf
    while i < len(a) and j < len(b):
        best = min(best, abs(a[i] - b[j]))
        if a[i] < b[j]:
            i += 1
        else:
            j += 1
    return int(best)


assert smallest_difference([1, 3, 15, 11, 2], [23, 127, 235, 19, 8]) == 3
for _ in range(200):
    x = random.sample(range(1000), 10); y = random.sample(range(1000), 10)
    assert smallest_difference(x, y) == min(abs(p - q) for p in x for q in y)
print("16.6 ok")

16.6 ok


## 16.7 Number Max
Return the larger of two numbers without `if`/`else` or any comparison operator. Use the sign bit of `a - b`. Watch for overflow in fixed-width ints: if a and b have different signs, the sign of `a` decides the answer. Here we emulate 32-bit ints.

In [8]:
def sign(x: int) -> int:
    """1 if x >= 0 else 0, using only bit ops on a 32-bit value."""
    return ((x >> 31) & 1) ^ 1


def to32(x: int) -> int:
    x &= 0xFFFFFFFF
    return x - (1 << 32) if x >> 31 else x


def number_max(a: int, b: int) -> int:
    c = to32(a - b)             # may overflow in 32-bit land
    sa, sb, sc = sign(a), sign(b), sign(c)
    use_sign_of_a = sa ^ sb     # different signs -> overflow possible -> trust sign(a)
    use_sign_of_c = use_sign_of_a ^ 1
    k = use_sign_of_a * sa + use_sign_of_c * sc
    q = k ^ 1
    return a * k + b * q


INT_MAX, INT_MIN = 2**31 - 1, -(2**31)
cases = [(3, 7), (7, 3), (-5, -2), (INT_MAX, -15), (INT_MIN, 3), (INT_MAX, INT_MIN), (4, 4)]
assert all(number_max(a, b) == max(a, b) for a, b in cases)
print("16.7 ok")

16.7 ok


## 16.8 English Int
Spell out an integer in English, e.g. `1234` → "One Thousand, Two Hundred Thirty Four". Split it into 3-digit chunks and attach thousand, million, or billion to each.

In [9]:
SMALLS = ["Zero", "One", "Two", "Three", "Four", "Five", "Six", "Seven", "Eight", "Nine", "Ten", "Eleven",
          "Twelve", "Thirteen", "Fourteen", "Fifteen", "Sixteen", "Seventeen", "Eighteen", "Nineteen"]
TENS = ["", "", "Twenty", "Thirty", "Forty", "Fifty", "Sixty", "Seventy", "Eighty", "Ninety"]
BIGS = ["", "Thousand", "Million", "Billion", "Trillion"]


def chunk_to_words(n: int) -> str:
    parts = []
    if n >= 100:
        parts += [SMALLS[n // 100], "Hundred"]
        n %= 100
    if n >= 20:
        parts.append(TENS[n // 10])
        n %= 10
    if n:
        parts.append(SMALLS[n])
    return " ".join(parts)


def english_int(n: int) -> str:
    if n == 0:
        return "Zero"
    if n < 0:
        return "Negative " + english_int(-n)
    chunks, i = [], 0
    while n:
        n, chunk = divmod(n, 1000)
        if chunk:
            chunks.append(f"{chunk_to_words(chunk)} {BIGS[i]}".strip())
        i += 1
    return ", ".join(reversed(chunks))


assert english_int(1234) == "One Thousand, Two Hundred Thirty Four"
assert english_int(-1_000_010) == "Negative One Million, Ten"
assert english_int(19) == "Nineteen" and english_int(100) == "One Hundred"
print(english_int(2_147_483_647))

Two Billion, One Hundred Forty Seven Million, Four Hundred Eighty Three Thousand, Six Hundred Forty Seven


## 16.9 Operations
Implement multiply, subtract, and divide for ints using **only addition** (and negation built from addition). Negate by adding `-1` or `+1` repeatedly. To make that faster, grow the step size by doubling (still only addition), which gives `O(log² n)`.

In [10]:
def negate(a: int) -> int:
    """Add +/-1 steps toward zero, doubling the step each time and resetting it on overshoot."""
    result = 0
    unit = 1 if a < 0 else -1
    delta = unit
    while a != 0:
        overshoots = a + delta != 0 and (a + delta > 0) != (a > 0)
        if overshoots:
            delta = unit
        result += delta
        a += delta
        delta += delta
    return result


def minus(a: int, b: int) -> int:
    return a + negate(b)


def abs_(a: int) -> int:
    return negate(a) if a < 0 else a


def multiply(a: int, b: int) -> int:
    if abs_(a) < abs_(b):
        return multiply(b, a)
    total = 0
    for _ in range(abs_(b)):
        total += a
    return negate(total) if b < 0 else total


def divide(a: int, b: int) -> int:
    """Integer division truncating toward zero."""
    if b == 0:
        raise ZeroDivisionError
    x, y = abs_(a), abs_(b)
    quotient, product = 0, 0
    while product + y <= x:
        product += y
        quotient += 1
    return negate(quotient) if (a < 0) != (b < 0) else quotient


for _ in range(300):
    p, q = random.randint(-300, 300), random.randint(-30, 30)
    assert negate(p) == -p and minus(p, q) == p - q and multiply(p, q) == p * q
    if q:
        assert divide(p, q) == int(p / q)
print("16.9 ok")

16.9 ok


## 16.10 Living People
Given people's birth and death years (all between 1900 and 2000), find the year with the most people alive. Someone who dies in a given year still counts as alive that year. Record +1 at each birth year and -1 at the year after each death, then take a running sum: `O(P + Y)`.

In [11]:
def living_people(people: list[tuple[int, int]], lo: int = 1900, hi: int = 2000) -> int:
    delta = [0] * (hi - lo + 2)
    for birth, death in people:
        delta[birth - lo] += 1
        delta[death - lo + 1] -= 1
    best_year, best, alive = lo, 0, 0
    for i, d in enumerate(delta[:-1]):
        alive += d
        if alive > best:
            best, best_year = alive, lo + i
    return best_year


def living_brute(people):
    return max(range(1900, 2001), key=lambda y: (sum(b <= y <= d for b, d in people), -y))


for _ in range(100):
    ppl = []
    for _ in range(30):
        b = random.randint(1900, 2000)
        ppl.append((b, random.randint(b, 2000)))
    assert living_people(ppl) == living_brute(ppl)
print("16.10 ok")

16.10 ok


## 16.11 Diving Board
Build a diving board from exactly K planks, where each plank is either `shorter` or `longer` in length. List every possible total length. Only the number of long planks matters (0…K), so there are K+1 lengths: `O(K)`.

In [12]:
def diving_board(k: int, shorter: int, longer: int) -> list[int]:
    if k == 0:
        return []
    if shorter == longer:
        return [k * shorter]
    return [(k - i) * shorter + i * longer for i in range(k + 1)]


assert diving_board(3, 1, 2) == [3, 4, 5, 6]
assert diving_board(4, 2, 2) == [8]
print("16.11 ok")

16.11 ok


## 16.12 XML Encoding
Encode XML compactly. Each tag and attribute name maps to an integer code, and `0` marks END. Grammar: `Element → Tag Attributes END Children END`, where `Attribute → Tag Value`. We parse with `xml.etree` and emit the tokens.

In [13]:
import xml.etree.ElementTree as ET


def encode_xml(xml: str, mapping: dict[str, int]) -> str:
    out: list[str] = []

    def encode(el: ET.Element) -> None:
        out.append(str(mapping[el.tag]))
        for k, v in el.attrib.items():
            out.extend([str(mapping[k]), v])
        out.append("0")
        if el.text and el.text.strip():
            out.append(el.text.strip())
        for child in el:
            encode(child)
        out.append("0")

    encode(ET.fromstring(xml))
    return " ".join(out)


xml = '<family lastName="Smith" state="CA"><person firstName="Jane">Some Message</person></family>'
codes = {"family": 1, "person": 2, "firstName": 3, "lastName": 4, "state": 5}
assert encode_xml(xml, codes) == "1 4 Smith 5 CA 0 2 3 Jane 0 Some Message 0 0"
print("16.12 ok")

16.12 ok


## 16.13 Bisect Squares
Given two squares on a plane, find a line that cuts both of them in half. Any line through a square's center halves it, so take the line through the two centers. Return the segment where that line meets the squares' outer edges. Assume the squares' sides are parallel to the x-axis.

In [14]:
@dataclass
class Square:
    left: float
    top: float     # y of the top edge (y grows downward, as on a screen)
    size: float

    @property
    def center(self) -> tuple[float, float]:
        return (self.left + self.size / 2, self.top + self.size / 2)

    def extend(self, mid1, mid2, size) -> tuple[float, float]:
        """Point where the line mid2->mid1 hits the edge of the square centred at mid1."""
        x1, y1 = mid1
        x2, y2 = mid2
        xdir = -1 if x1 < x2 else 1
        ydir = -1 if y1 < y2 else 1
        if x1 == x2:
            return (x1, y1 + ydir * size / 2)
        slope = (y1 - y2) / (x1 - x2)
        if abs(slope) == 1:
            return (x1 + xdir * size / 2, y1 + ydir * size / 2)
        if abs(slope) < 1:
            x = x1 + xdir * size / 2
            return (x, slope * (x - x1) + y1)
        y = y1 + ydir * size / 2
        return ((y - y1) / slope + x1, y)


def bisect_squares(a: Square, b: Square) -> tuple[tuple[float, float], tuple[float, float]]:
    c1, c2 = a.center, b.center
    pts = [a.extend(c1, c2, a.size), a.extend(c2, c1, -a.size),
           b.extend(c1, c2, -b.size), b.extend(c2, c1, b.size)]
    # the segment's endpoints are the two points furthest apart
    return max(((p, q) for p in pts for q in pts), key=lambda pq: math.dist(*pq))


seg = bisect_squares(Square(0, 0, 2), Square(4, 1, 2))
(x1, y1), (x2, y2) = seg
for sq in (Square(0, 0, 2), Square(4, 1, 2)):
    cx, cy = sq.center
    assert abs((x2 - x1) * (cy - y1) - (y2 - y1) * (cx - x1)) < 1e-9  # center lies on the line
print("16.13 ok, segment:", seg)

16.13 ok, segment: ((0.0, 0.75), (6.0, 2.25))


## 16.14 Best Line
Given a set of points, find the line that passes through the most of them. For every pair, key the line by a normalized (slope, intercept). Use exact fractions instead of the book's floating-point epsilon: `O(N²)`.

In [15]:
def best_line(points: list[tuple[int, int]]) -> tuple[tuple, int]:
    lines: dict[tuple, set[int]] = defaultdict(set)
    for i, (x1, y1) in enumerate(points):
        for j in range(i + 1, len(points)):
            x2, y2 = points[j]
            if x1 == x2:
                key = ("vertical", Fraction(x1))
            else:
                slope = Fraction(y2 - y1, x2 - x1)
                key = (slope, y1 - slope * x1)
            lines[key] |= {i, j}
    key, members = max(lines.items(), key=lambda kv: len(kv[1]))
    return key, len(members)


pts = [(0, 0), (1, 1), (2, 2), (3, 3), (1, 0), (2, 5), (0, 3), (3, 0)]
key, cnt = best_line(pts)
assert cnt == 4 and key == (Fraction(1), Fraction(0))
print("16.14 ok:", key, cnt)

16.14 ok: (Fraction(1, 1), Fraction(0, 1)) 4


## 16.15 Master Mind
The computer hides 4 slots, each colored R, G, B, or Y. For a guess, report the **hits** (right color in the right slot) and **pseudo-hits** (right color, wrong slot). Count hits first, then use the color frequencies of the unmatched slots to count pseudo-hits.

In [16]:
def master_mind(solution: str, guess: str) -> tuple[int, int]:
    hits = sum(s == g for s, g in zip(solution, guess))
    sol_left = Counter(s for s, g in zip(solution, guess) if s != g)
    guess_left = Counter(g for s, g in zip(solution, guess) if s != g)
    pseudo = sum((sol_left & guess_left).values())
    return hits, pseudo


assert master_mind("RGBY", "GGRR") == (1, 1)
assert master_mind("RRRR", "RRRR") == (4, 0)
assert master_mind("RGBY", "YBGR") == (0, 4)
print("16.15 ok")

16.15 ok


## 16.16 Sub Sort
Given an array, find indices m and n such that sorting only `a[m..n]` makes the whole array sorted. Minimize n - m. In `O(N)`: find the sorted left prefix and right suffix, take the min and max of the middle, then shrink the prefix and suffix until every value fits.

In [17]:
def sub_sort(a: list[int]) -> tuple[int, int] | None:
    n = len(a)
    left_end = 0
    while left_end + 1 < n and a[left_end + 1] >= a[left_end]:
        left_end += 1
    if left_end == n - 1:
        return None
    right_start = n - 1
    while right_start > 0 and a[right_start - 1] <= a[right_start]:
        right_start -= 1
    lo = min(a[left_end + 1:right_start + 1] or [math.inf])
    hi = max(a[left_end:right_start] or [-math.inf])
    lo = min(lo, a[right_start])
    hi = max(hi, a[left_end])
    m = left_end
    while m >= 0 and a[m] > lo:
        m -= 1
    k = right_start
    while k < n and a[k] < hi:
        k += 1
    return m + 1, k - 1


def sub_sort_brute(a):
    s = sorted(a)
    diff = [i for i in range(len(a)) if a[i] != s[i]]
    return (diff[0], diff[-1]) if diff else None


assert sub_sort([1, 2, 4, 7, 10, 11, 7, 12, 6, 7, 16, 18, 19]) == (3, 9)
for _ in range(500):
    arr = [random.randint(0, 9) for _ in range(random.randint(1, 10))]
    assert sub_sort(arr) == sub_sort_brute(arr), arr
print("16.16 ok")

16.16 ok


## 16.17 Contiguous Sequence
Find the contiguous subarray with the largest sum (Kadane's algorithm, `O(N)`).

In [18]:
def max_subarray(a: list[int]) -> int:
    best = cur = a[0]
    for x in a[1:]:
        cur = max(x, cur + x)
        best = max(best, cur)
    return best


assert max_subarray([2, -8, 3, -2, 4, -10]) == 5
assert max_subarray([-3, -1, -2]) == -1
print("16.17 ok")

16.17 ok


## 16.18 Pattern Matching
A pattern made of `a`s and `b`s (e.g. `aabab`) matches a value string (e.g. `catcatgocatgo`) if the letters can be mapped to substrings. Try every possible length for the main letter. That fixes the length of the other letter, so each candidate is checked in `O(n)`, for `O(n²)` total.

In [19]:
def pattern_matches(pattern: str, value: str) -> bool:
    if not pattern:
        return not value
    main = pattern[0]
    alt = "b" if main == "a" else "a"
    n_main, n_alt = pattern.count(main), pattern.count(alt)
    first_alt = pattern.find(alt)
    for main_len in range(len(value) // n_main + 1):
        rem = len(value) - main_len * n_main
        if n_alt == 0:
            if rem != 0:
                continue
            alt_len = 0
        else:
            if rem % n_alt:
                continue
            alt_len = rem // n_alt
        main_s = value[:main_len]
        alt_s = value[first_alt * main_len:first_alt * main_len + alt_len] if n_alt else ""
        if "".join(main_s if p == main else alt_s for p in pattern) == value:
            return True
    return False


assert pattern_matches("aabab", "catcatgocatgo")
assert pattern_matches("a", "catcatgocatgo") and pattern_matches("b", "catcatgocatgo")
assert pattern_matches("ab", "catcatgocatgo")
assert not pattern_matches("aabab", "catcatgocatg")
assert pattern_matches("bba", "gogocat")
print("16.18 ok")

16.18 ok


## 16.19 Pond Sizes
In a grid of land heights, a 0 is water, and a pond is a region of water joined horizontally, vertically, or diagonally. Return the sizes of all ponds using a BFS flood fill.

In [20]:
def pond_sizes(land: list[list[int]]) -> list[int]:
    rows, cols = len(land), len(land[0])
    seen = [[False] * cols for _ in range(rows)]
    sizes = []
    for r in range(rows):
        for c in range(cols):
            if land[r][c] == 0 and not seen[r][c]:
                seen[r][c] = True
                q, size = deque([(r, c)]), 0
                while q:
                    cr, cc = q.popleft()
                    size += 1
                    for dr in (-1, 0, 1):
                        for dc in (-1, 0, 1):
                            nr, nc = cr + dr, cc + dc
                            if 0 <= nr < rows and 0 <= nc < cols and land[nr][nc] == 0 and not seen[nr][nc]:
                                seen[nr][nc] = True
                                q.append((nr, nc))
                sizes.append(size)
    return sizes


assert sorted(pond_sizes([[0, 2, 1, 0], [0, 1, 0, 1], [1, 1, 0, 1], [0, 1, 0, 1]])) == [1, 2, 4]
print("16.19 ok")

16.19 ok


## 16.20 T9
On old phone keypads each digit maps to letters. Given a digit sequence, return every valid word it could spell. Precompute a map from digit-string to words, so each lookup is `O(1)`.

In [21]:
KEYPAD = {"2": "abc", "3": "def", "4": "ghi", "5": "jkl", "6": "mno", "7": "pqrs", "8": "tuv", "9": "wxyz"}
LETTER_TO_DIGIT = {ch: d for d, letters in KEYPAD.items() for ch in letters}


def build_t9_index(words: list[str]) -> dict[str, list[str]]:
    index: dict[str, list[str]] = defaultdict(list)
    for w in words:
        index["".join(LETTER_TO_DIGIT[c] for c in w)].append(w)
    return index


WORDS = ["tree", "used", "vsed", "apple", "bat", "cat", "act"]
t9 = build_t9_index(WORDS)
assert t9["8733"] == ["tree", "used", "vsed"] and t9["228"] == ["bat", "cat", "act"]
print("16.20 ok")

16.20 ok


## 16.21 Sum Swap
Given two int arrays, find one value from each to swap so that the arrays end up with equal sums. We need `a - b = (sumA - sumB) / 2`, so look up each candidate in a hash set: `O(A + B)`.

In [22]:
def sum_swap(a: list[int], b: list[int]) -> tuple[int, int] | None:
    diff = sum(a) - sum(b)
    if diff % 2:
        return None
    target = diff // 2
    bs = set(b)
    for x in a:
        if x - target in bs:
            return x, x - target
    return None


A, B = [4, 1, 2, 1, 1, 2], [3, 6, 3, 3]
x, y = sum_swap(A, B)
assert sum(A) - x + y == sum(B) - y + x
print("16.21 ok:", (x, y))

16.21 ok: (4, 6)


## 16.22 Langton's Ant
An ant on an infinite grid of white and black cells. On a white cell it flips the color, turns right 90°, and moves forward. On a black cell it flips, turns left, and moves. Simulate K moves and print the final grid. A set of black cells gives an infinite grid for free.

In [23]:
def langtons_ant(k: int) -> str:
    black: set[tuple[int, int]] = set()
    pos, heading = (0, 0), 0  # 0=up,1=right,2=down,3=left
    moves = [(-1, 0), (0, 1), (1, 0), (0, -1)]
    for _ in range(k):
        if pos in black:
            black.remove(pos); heading = (heading - 1) % 4
        else:
            black.add(pos); heading = (heading + 1) % 4
        pos = (pos[0] + moves[heading][0], pos[1] + moves[heading][1])
    cells = black | {pos}
    rmin, rmax = min(r for r, _ in cells), max(r for r, _ in cells)
    cmin, cmax = min(c for _, c in cells), max(c for _, c in cells)
    arrow = "^>v<"[heading]
    return "\n".join("".join(arrow if (r, c) == pos else "X" if (r, c) in black else "_"
                             for c in range(cmin, cmax + 1)) for r in range(rmin, rmax + 1))


print(langtons_ant(11_000)[:600])  # after ~10k steps the famous 'highway' emerges

_____________________________XX__XXXXXXXXXXXX__XX__________________
____________________________X__XXXX__________X__XX_________________
___________________________XXX___XX____________XX_X________________
___________________________X_X__X_________X__X____X________________
_______________________XX__XX_X_X_________XXX_______X______________
____________________XXX_X__X___X_____X_____XX_XX__XXX______________
_____________________X_X__XXX__XX_XXXX_XX___X_X__X_XX__XX__________
_____________________X_XXX_XX__X_XX__XXX_X_X_____XXX___XXX_________
___________________X_____X___XXXXX_X_X__XXXX__X___XXX_X_


## 16.23 Rand7 from Rand5
Build `rand7()` from `rand5()`. Every value must be equally likely. Use `5*rand5() + rand5()`, which is uniform over 0..24, and reject values ≥ 21. What's left is uniform over 0..20, so take it mod 7.

In [24]:
def rand5() -> int:
    return random.randrange(5)


def rand7() -> int:
    while True:
        n = 5 * rand5() + rand5()
        if n < 21:
            return n % 7


hist = Counter(rand7() for _ in range(70_000))
assert sorted(hist) == list(range(7)) and all(9_300 < v < 10_700 for v in hist.values())
print("16.23 ok", dict(sorted(hist.items())))

16.23 ok {0: 10219, 1: 10052, 2: 9953, 3: 9885, 4: 10092, 5: 9942, 6: 9857}


## 16.24 Pairs with Sum
Find every pair of ints in an array that sums to a given value. Use a hash map of unmatched counts, so duplicates are handled properly: `O(N)`.

In [25]:
def pairs_with_sum(a: list[int], target: int) -> list[tuple[int, int]]:
    unpaired: Counter[int] = Counter()
    pairs = []
    for x in a:
        comp = target - x
        if unpaired[comp] > 0:
            pairs.append((comp, x))
            unpaired[comp] -= 1
        else:
            unpaired[x] += 1
    return pairs


assert sorted(tuple(sorted(p)) for p in pairs_with_sum([5, 3, 7, 0, 1, 4, 2, 5], 5)) == [(0, 5), (1, 4), (2, 3)]
assert pairs_with_sum([5, 5, 5], 10) == [(5, 5)]
print("16.24 ok")

16.24 ok


## 16.25 LRU Cache
Build an LRU cache: once it's full, inserting evicts the least recently used item. The classic design is a hash map plus a doubly linked list, which we implement by hand. `OrderedDict` gives you the same thing in a few lines. (`functools.lru_cache` is the decorator form.)

In [26]:
class _DNode:
    __slots__ = ("key", "val", "prev", "next")

    def __init__(self, key=None, val=None) -> None:
        self.key, self.val = key, val
        self.prev: _DNode | None = None
        self.next: _DNode | None = None


class LRUCache:
    def __init__(self, capacity: int) -> None:
        self.cap = capacity
        self.map: dict = {}
        self.head, self.tail = _DNode(), _DNode()  # sentinels: head.next = most recent
        self.head.next, self.tail.prev = self.tail, self.head

    def _unlink(self, n: _DNode) -> None:
        n.prev.next, n.next.prev = n.next, n.prev

    def _push_front(self, n: _DNode) -> None:
        n.prev, n.next = self.head, self.head.next
        self.head.next.prev = n
        self.head.next = n

    def get(self, key):
        n = self.map.get(key)
        if n is None:
            return None
        self._unlink(n); self._push_front(n)
        return n.val

    def put(self, key, val) -> None:
        if key in self.map:
            self._unlink(self.map.pop(key))
        elif len(self.map) == self.cap:
            lru = self.tail.prev
            self._unlink(lru)
            del self.map[lru.key]
        n = _DNode(key, val)
        self.map[key] = n
        self._push_front(n)


class LRUCacheOD(OrderedDict):
    def __init__(self, capacity: int) -> None:
        super().__init__(); self.cap = capacity

    def get(self, key):  # type: ignore[override]
        if key not in self: return None
        self.move_to_end(key); return self[key]

    def put(self, key, val) -> None:
        self[key] = val; self.move_to_end(key)
        if len(self) > self.cap: self.popitem(last=False)


for C in (LRUCache, LRUCacheOD):
    c = C(2)
    c.put(1, "a"); c.put(2, "b"); c.get(1); c.put(3, "c")
    assert c.get(2) is None and c.get(1) == "a" and c.get(3) == "c"
print("16.25 ok")

16.25 ok


## 16.26 Calculator
Evaluate an expression of positive integers and `+ - * /` (no parentheses), e.g. `2*3+5/6*3+15`. A single left-to-right pass with an operand stack: apply `*` and `/` immediately, save `+` and `-` for the end. Use `Fraction` for exact division.

In [27]:
def calculate(expr: str) -> Fraction:
    tokens = re.findall(r"\d+|[-+*/]", expr.replace(" ", ""))
    stack: list[Fraction] = [Fraction(int(tokens[0]))]
    for op, num in zip(tokens[1::2], tokens[2::2]):
        n = Fraction(int(num))
        match op:
            case "*": stack[-1] *= n
            case "/": stack[-1] /= n
            case "+": stack.append(n)
            case "-": stack.append(-n)
    return sum(stack, Fraction(0))


assert calculate("2*3+5/6*3+15") == Fraction(47, 2)
assert calculate("2-6-7*8/2+5") == -27
print("16.26 ok:", float(calculate("2*3+5/6*3+15")))

16.26 ok: 23.5
